# 🚀 Multi-Source RAG — LangGraph + Groq + Gemini Embeddings

**Architecture:** LangGraph `StateGraph` with 4 routing paths, cost/latency tracking, and real token counting

**Stack:**
- 🤖 **LLM (Routing + Synthesis):** Groq Llama 3.3 70B — ultra-fast inference, no tight rate limits
- 🔢 **Embeddings:** Google Gemini `gemini-embedding-001` → local FAISS vector store
- 🕸️ **Orchestration:** LangGraph `StateGraph`

**Why LangGraph over AgentExecutor?**
- Explicit state visible at every step (no hidden chain reasoning)
- Node-level tracing in LangSmith — see exactly how long each step takes
- Conditional routing is code, not prompt magic — easy to debug and iterate
- Easy to extend with checkpointing, memory, and custom nodes

```
START → router_node → [wikipedia | arxiv | langsmith | direct_answer] → answer_node → END
```

## Cell 1: Setup & Environment

In [ ]:
import sys, os
os.environ['USER_AGENT'] = 'MultiSourceRAG/1.0'
sys.path.insert(0, '../src')
sys.path.insert(0, '../eval')

from dotenv import load_dotenv
load_dotenv('../.env')

from config import LLM, SYNTHESIS_LLM, TRACING_ENABLED, get_text
print(f'✅ Environment loaded')
print(f'   LLM model:         Groq Llama 3.3 70B')
print(f'   Embeddings:        Gemini gemini-embedding-001')
print(f'   LangSmith tracing: {TRACING_ENABLED}')

# Quick model smoke test
r = LLM.invoke('Say OK')
print(f'\n✅ LLM test: "{get_text(r.content)}"')

## Cell 2: Build the Graph & Visualize Topology

In [ ]:
from graph import graph, run_query

# Show the graph topology as a Mermaid diagram
try:
    display(graph.get_graph().draw_mermaid_png())
except Exception:
    # Fallback to raw Mermaid text
    print(graph.get_graph().draw_mermaid())

## Cell 3: Token Tracking Verification

In [ ]:
# Verify real token counts are captured from Groq responses
from tracking import extract_token_usage
from langchain_core.messages import HumanMessage

resp = LLM.invoke([HumanMessage(content='Explain RAG in one sentence.')])
inp, out = extract_token_usage(resp)

print(f'✅ Token tracking (Groq dict format):')
print(f'   Input tokens:  {inp}')
print(f'   Output tokens: {out}')
print(f'   Response:      "{get_text(resp.content)}"')

## Cell 4: Arxiv Tool Verification (Fixed API)

In [ ]:
# Verify the fixed arxiv.Client() API returns real paper results
from tools import _arxiv_search

result = _arxiv_search('attention is all you need transformer')
print(f'✅ Arxiv tool returns {len(result)} chars of real paper data:')
print(result[:400])

## Cell 5: Single Query Demos — All 4 Routes

In [ ]:
import time

demo_queries = [
    ('Who is Marie Curie?',                            'wikipedia'),
    ('Recent research on attention mechanisms',         'arxiv'),
    ('How does LangSmith tracing work?',               'langsmith_search'),
    ('What is 15 * 7?',                                'direct_answer'),
]

results = []
for query, expected in demo_queries:
    if results:
        time.sleep(3)  # brief pause between queries
    r = run_query(query, verbose=False)
    ok = '✅' if r['route'] == expected else '❌'
    print(f'{ok} [{r["route"]:18}] {query[:50]}')
    print(f'   Latency: {r["total_latency_ms"]:.0f}ms | Cost: ${r["total_cost_usd"]:.6f}')
    print(f'   Answer: {r["answer"][:120]}...')
    print()
    results.append(r)

## Cell 6: Cost & Latency Analysis

In [ ]:
# Show the cost/latency breakdown from the 4 demo queries above
print(f'\n{"Route":<22} {"Latency":>10} {"Cost":>12}')
print('─' * 46)
for r in results:
    print(f'{r["route"]:<22} {r["total_latency_ms"]:>9.0f}ms {r["total_cost_usd"]:>11.6f}$')

total_cost = sum(r['total_cost_usd'] for r in results)
avg_lat = sum(r['total_latency_ms'] for r in results) / len(results)
print(f'─' * 46)
print(f'{"AVERAGE":<22} {avg_lat:>9.0f}ms {total_cost/len(results):>11.6f}$')
print(f'\n💡 direct_answer is fastest: no external tool call')
print(f'💡 wikipedia is slowest: Wikipedia API network latency')
print(f'💡 Total for 4 demo queries: ${total_cost:.6f}')

## Cell 7: Load Saved Evaluation Results

In [ ]:
import json, glob, os

# Load the most recent saved evaluation result
results_dir = '../results'
eval_files = sorted(glob.glob(os.path.join(results_dir, 'eval_groq*.json')))

if eval_files:
    with open(eval_files[-1]) as f:
        saved = json.load(f)

    print(f'📂 Loaded: {os.path.basename(eval_files[-1])}')
    print(f'\n📊 Overall Routing Accuracy: {saved["correct"]}/{saved["total"]} = {saved["overall_accuracy"]:.1%}')
    print(f'   Avg Latency:  {saved["avg_latency_ms"]:.0f}ms')
    print(f'   Avg Cost:     ${saved["avg_cost_usd"]:.6f}')
    print(f'   Total Cost:   ${saved["total_cost_usd"]:.5f}')
    print(f'   Retrieval HR: {saved["avg_retrieval_hit_rate"]:.1%}')

    print(f'\n  Per-Tool Breakdown:')
    print(f'  {"Tool":<22} {"Precision":>10} {"Recall":>8} {"Avg Lat":>10}')
    print(f'  {"─"*52}')
    for tool, m in saved['per_tool_metrics'].items():
        print(f'  {tool:<22} {m["precision"]:>10.1%} {m["recall"]:>8.1%} {m["avg_latency_ms"]:>9.0f}ms')
else:
    print('No saved results found. Run: python eval/run_eval.py --label groq_baseline --delay 5')

## Cell 8: Interactive Query — Try Your Own

In [ ]:
# ✏️ Change this query to anything you want!
your_query = "What are the main components of a RAG system?"

result = run_query(your_query)
print(f'\n{"="*60}')
print(f'Query:   {your_query}')
print(f'Route:   {result["route"]}')
print(f'Latency: {result["total_latency_ms"]:.0f}ms')
print(f'Cost:    ${result["total_cost_usd"]:.6f}')
print(f'\nAnswer:\n{result["answer"]}')